In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import tkinter as tk
from tkinter import ttk
from scipy.integrate import solve_ivp
import threading
import time

class FullyLinkedTankSystem:
    def __init__(self, root):
        self.root = root
        self.root.title("Fully Linked Three-Tank System - Real-time Control")
        self.root.geometry("1600x1000")
        
        # Default parameters
        self.params = {
            'g': 9.81, 'A1': 1.0, 'A2': 0.8, 'A3': 0.8,
            'a2': 0.01, 'a3': 0.01, 'a4': 0.008, 'a5': 0.008,
            'Smax': 1.0, 'u': 0.05
        }
        
        self.initial_conditions = [0.1, 0.05, 0.05]
        self.simulation_time = (0, 100)
        self.current_time = 0
        self.solution = None
        self.is_animating = False
        self.animation_speed = 1.0
        self.auto_recalculate = True 
        
        # Initialize sliders dictionary
        self.sliders = {}
        
        self.setup_gui()
        self.solve_system()
        self.update_display()
    
    def tank_system(self, t, h):
        h1, h2, h3 = h
        h1, h2, h3 = max(h1, 0), max(h2, 0), max(h3, 0)
        
        dhdt = np.zeros(3)
        dhdt[0] = (self.params['u'] - self.params['Smax'] * self.params['a2'] * np.sqrt(2 * self.params['g'] * h1) - self.params['Smax'] * self.params['a3'] * np.sqrt(2 * self.params['g'] * h1)) / self.params['A1']
        dhdt[1] = (self.params['Smax'] * self.params['a2'] * np.sqrt(2 * self.params['g'] * h1) - self.params['Smax'] * self.params['a4'] * np.sqrt(2 * self.params['g'] * h2)) / self.params['A2']
        dhdt[2] = (self.params['Smax'] * self.params['a3'] * np.sqrt(2 * self.params['g'] * h1) - self.params['Smax'] * self.params['a5'] * np.sqrt(2 * self.params['g'] * h3)) / self.params['A3']
        
        return dhdt
    
    def solve_system(self):
        try:
            t_eval = np.linspace(self.simulation_time[0], self.simulation_time[1], 1000)
            self.solution = solve_ivp(self.tank_system, self.simulation_time, self.initial_conditions, t_eval=t_eval, method='RK23')
            return self.solution
        except Exception as e:
            print(f"Error solving system: {e}")
            return None
    
    def setup_gui(self):
        # Configure main grid
        self.root.columnconfigure(0, weight=0)
        self.root.columnconfigure(1, weight=1)
        self.root.rowconfigure(0, weight=1)
        
        # Create frames with better proportions
        control_frame = ttk.Frame(self.root, padding="10")
        control_frame.grid(row=0, column=0, sticky="nsew", padx=5, pady=5)
        
        display_frame = ttk.Frame(self.root, padding="10")
        display_frame.grid(row=0, column=1, sticky="nsew", padx=5, pady=5)
        
        bottom_frame = ttk.Frame(self.root, padding="10")
        bottom_frame.grid(row=1, column=0, columnspan=2, sticky="ew", padx=5, pady=5)
        
        self.setup_controls(control_frame)
        self.setup_matplotlib(display_frame)
        self.setup_animation_controls(bottom_frame)
    
    def setup_controls(self, parent):
        # Configure control frame grid
        parent.columnconfigure(0, weight=1)
        
        title = ttk.Label(parent, text="💧 HYDRAULIC SYSTEM WITH THREE TANKS", 
                         font=('Arial', 14, 'bold'), foreground='blue')
        title.grid(row=0, column=0, pady=10, sticky='ew')
        
        # Auto-recalculate
        self.auto_recalc_var = tk.BooleanVar(value=True)
        auto_recalc_cb = ttk.Checkbutton(parent, text="🔄 AUTO-RECALCULATE (Instant Updates)", 
                                       variable=self.auto_recalc_var, command=self.on_auto_recalc_change)
        auto_recalc_cb.grid(row=1, column=0, pady=5, sticky='w')
        
        # Create notebook for better organization
        notebook = ttk.Notebook(parent)
        notebook.grid(row=2, column=0, sticky='nsew', pady=10)
        
        # Flow control tab
        flow_frame = ttk.Frame(notebook, padding="10")
        notebook.add(flow_frame, text="💧 FLOW CONTROL")
        self.setup_flow_controls(flow_frame)
        
        # Valve control tab
        valve_frame = ttk.Frame(notebook, padding="10")
        notebook.add(valve_frame, text="🔧 VALVES")
        self.setup_valve_controls(valve_frame)
        
        # Tank control tab
        tank_frame = ttk.Frame(notebook, padding="10")
        notebook.add(tank_frame, text="📦 TANKS")
        self.setup_tank_controls(tank_frame)
        
        # Initial conditions tab
        init_frame = ttk.Frame(notebook, padding="10")
        notebook.add(init_frame, text="🎯 INITIAL")
        self.setup_initial_controls(init_frame)
        
        # Control buttons
        button_frame = ttk.Frame(parent)
        button_frame.grid(row=3, column=0, pady=20, sticky='ew')
        
        ttk.Button(button_frame, text="🔄 FORCE RECALCULATE", 
                  command=self.force_recalculate, style='Accent.TButton').pack(side='left', padx=5)
        ttk.Button(button_frame, text="🔄 RESET ALL", 
                  command=self.reset_all).pack(side='left', padx=5)
        
        # Current values display
        values_frame = ttk.LabelFrame(parent, text="📊 REAL-TIME SYSTEM STATE", padding="10")
        values_frame.grid(row=4, column=0, sticky='ew', pady=10)
        
        self.values_text = tk.Text(values_frame, height=15, width=40, font=('Courier', 9))
        scrollbar = ttk.Scrollbar(values_frame, orient="vertical", command=self.values_text.yview)
        self.values_text.configure(yscrollcommand=scrollbar.set)
        
        self.values_text.pack(side='left', fill='both', expand=True)
        scrollbar.pack(side='right', fill='y')
    
    def setup_flow_controls(self, parent):
        parent.columnconfigure(0, weight=1)
        self.create_slider(parent, "Input Flow (u)", 'u', 0.01, 0.1, 0.05, 0)
        
        # Flow visualization info
        info_label = ttk.Label(parent, text="💡 Flow: u → Tank 1 → Tanks 2 & 3 → Out", 
                              font=('Arial', 9), foreground='gray')
        info_label.grid(row=1, column=0, pady=10, sticky='w')
    
    def setup_valve_controls(self, parent):
        parent.columnconfigure(0, weight=1)
        
        # Valve diagram explanation
        diagram_text = """
        Valve Connections:
        • a2: Tank 1 → Tank 2 (Left pipe)
        • a3: Tank 1 → Tank 3 (Right pipe)  
        • a4: Tank 2 Out (Bottom)
        • a5: Tank 3 Out (Bottom)
        """
        diagram_label = ttk.Label(parent, text=diagram_text, font=('Courier', 9), 
                                justify='left', background='#f0f0f0', padding=5)
        diagram_label.grid(row=0, column=0, columnspan=2, pady=5, sticky='w')
        
        self.create_slider(parent, "Valve a2 (T1→T2)", 'a2', 0.005, 0.02, 0.01, 1)
        self.create_slider(parent, "Valve a3 (T1→T3)", 'a3', 0.005, 0.02, 0.01, 2)
        self.create_slider(parent, "Valve a4 (T2 Out)", 'a4', 0.004, 0.015, 0.008, 3)
        self.create_slider(parent, "Valve a5 (T3 Out)", 'a5', 0.004, 0.015, 0.008, 4)
    
    def setup_tank_controls(self, parent):
        parent.columnconfigure(0, weight=1)
        self.create_slider(parent, "Tank 1 Area (A1)", 'A1', 0.5, 2.0, 1.0, 0)
        self.create_slider(parent, "Tank 2 Area (A2)", 'A2', 0.5, 2.0, 0.8, 1)
        self.create_slider(parent, "Tank 3 Area (A3)", 'A3', 0.5, 2.0, 0.8, 2)
    
    def setup_initial_controls(self, parent):
        parent.columnconfigure(0, weight=1)
        self.create_slider(parent, "Initial h1", 'init_h1', 0.0, 1.0, 0.1, 0)
        self.create_slider(parent, "Initial h2", 'init_h2', 0.0, 1.0, 0.05, 1)
        self.create_slider(parent, "Initial h3", 'init_h3', 0.0, 1.0, 0.05, 2)
    
    def create_slider(self, parent, label, param, min_val, max_val, default, row):
        """Create a slider with label and value display"""
        # Main frame for this slider
        slider_frame = ttk.Frame(parent)
        slider_frame.grid(row=row, column=0, sticky='ew', pady=3)
        slider_frame.columnconfigure(1, weight=1)
        
        # Label
        ttk.Label(slider_frame, text=label, width=20, anchor='w').grid(row=0, column=0, sticky='w')
        
        # Slider
        var = tk.DoubleVar(value=default)
        slider = ttk.Scale(slider_frame, from_=min_val, to=max_val, orient='horizontal', 
                          variable=var, command=lambda v, p=param: self.on_slider_change(p, v))
        slider.grid(row=0, column=1, sticky='ew', padx=5)
        
        # Value label with better formatting
        value_label = ttk.Label(slider_frame, text=f"{default:.3f}", width=8)
        value_label.grid(row=0, column=2, padx=5)
        
        # Store references
        self.sliders[param] = {'var': var, 'slider': slider, 'label': value_label}
    
    def setup_matplotlib(self, parent):
        """Setup matplotlib figures in Tkinter"""
        # Create figure with subplots
        self.fig = plt.figure(figsize=(14, 10))
        gs = self.fig.add_gridspec(2, 2, width_ratios=[1.2, 1])
        
        self.ax_tanks = self.fig.add_subplot(gs[:, 0])  # Tanks (left)
        self.ax_levels = self.fig.add_subplot(gs[0, 1])  # All levels (top right)
        self.ax_h1 = self.fig.add_subplot(gs[1, 1])     # h1 only (bottom right)
        
        # Embed in Tkinter
        self.canvas = FigureCanvasTkAgg(self.fig, parent)
        self.canvas.get_tk_widget().pack(fill='both', expand=True)
    
    def setup_animation_controls(self, parent):
        """Setup animation controls at bottom"""
        # Animation title
        title = ttk.Label(parent, text="⏯️ ANIMATION CONTROLS", 
                         font=('Arial', 12, 'bold'), foreground='green')
        title.pack(pady=5)
        
        # Main animation controls
        control_frame = ttk.Frame(parent)
        control_frame.pack(pady=10, fill='x')
        
        # Start/Stop button
        self.animate_button = ttk.Button(control_frame, text="▶ START ANIMATION", 
                                        command=self.toggle_animation, style='Accent.TButton')
        self.animate_button.pack(side='left', padx=10)
        
        # Speed control
        ttk.Label(control_frame, text="Speed:").pack(side='left', padx=10)
        self.speed_var = tk.DoubleVar(value=1.0)
        speed_slider = ttk.Scale(control_frame, from_=0.1, to=3.0, orient='horizontal',
                                variable=self.speed_var, command=self.on_speed_change)
        speed_slider.pack(side='left', padx=5)
        
        self.speed_label = ttk.Label(control_frame, text="1.0x")
        self.speed_label.pack(side='left', padx=5)
        
        # Time display
        self.time_display = ttk.Label(control_frame, text="Time: 0.0 s", font=('Arial', 10, 'bold'))
        self.time_display.pack(side='left', padx=20)
        
        # Manual time control
        ttk.Label(control_frame, text="Manual Time:").pack(side='left', padx=10)
        self.manual_time_var = tk.DoubleVar(value=0)
        manual_time_slider = ttk.Scale(control_frame, from_=0, to=100, orient='horizontal',
                                      variable=self.manual_time_var, command=self.on_manual_time_change)
        manual_time_slider.pack(side='left', padx=5)
        
        # Progress bar
        progress_frame = ttk.Frame(parent)
        progress_frame.pack(fill='x', padx=20, pady=5)
        
        ttk.Label(progress_frame, text="Progress:").pack(side='left')
        self.progress_var = tk.DoubleVar(value=0)
        self.progress = ttk.Progressbar(progress_frame, variable=self.progress_var, maximum=100)
        self.progress.pack(side='left', fill='x', expand=True, padx=5)
    
    def on_auto_recalc_change(self):
        """Handle auto-recalculate checkbox change"""
        self.auto_recalculate = self.auto_recalc_var.get()
        status = "ENABLED" if self.auto_recalculate else "DISABLED"
        print(f"🔄 Auto-recalculate: {status}")
    
    def on_slider_change(self, param, value):
        """Handle slider changes - AUTOMATICALLY UPDATE EVERYTHING"""
        value_float = float(value)
        self.sliders[param]['label'].config(text=f"{value_float:.3f}")
        
        # Update parameters or initial conditions
        if param in self.params:
            self.params[param] = value_float
        elif param.startswith('init_'):
            tank_idx = int(param[-1]) - 1
            self.initial_conditions[tank_idx] = value_float
        
        # AUTO-RECALCULATE if enabled
        if self.auto_recalculate:
            self.auto_recalculate_system()
        else:
            self.update_values_display()
    
    def auto_recalculate_system(self):
        """Automatically recalculate system with brief delay"""
        # Stop any pending recalculations
        if hasattr(self, 'recalc_job'):
            self.root.after_cancel(self.recalc_job)
        
        # Schedule recalculation after brief delay (debouncing)
        self.recalc_job = self.root.after(300, self.force_recalculate)  # 300ms delay
    
    def on_speed_change(self, value):
        """Handle animation speed change"""
        self.animation_speed = float(value)
        self.speed_label.config(text=f"{self.animation_speed:.1f}x")
    
    def on_manual_time_change(self, value):
        """Handle manual time slider change"""
        self.current_time = float(value)
        self.manual_time_var.set(self.current_time)
        self.update_display()
    
    def toggle_animation(self):
        """Start or stop animation"""
        if not self.is_animating:
            self.start_animation()
        else:
            self.stop_animation()
    
    def start_animation(self):
        """Start the animation"""
        self.is_animating = True
        self.animate_button.config(text="⏹ STOP ANIMATION")
        self.animation_thread = threading.Thread(target=self.run_animation, daemon=True)
        self.animation_thread.start()
    
    def stop_animation(self):
        """Stop the animation"""
        self.is_animating = False
        self.animate_button.config(text="▶ START ANIMATION")
    
    def run_animation(self):
        """Run animation in separate thread"""
        frame_time = 0.05 / self.animation_speed
        
        while self.is_animating and self.solution is not None:
            # Increment time
            self.current_time += 0.5 * self.animation_speed
            
            # Check if we reached the end
            if self.current_time >= self.simulation_time[1]:
                self.current_time = 0  # Loop back to start
            
            # Update manual time slider
            self.manual_time_var.set(self.current_time)
            
            # Update display in main thread
            self.root.after(0, self.update_animation_display)
            
            # Wait for next frame
            time.sleep(frame_time)
    
    def update_animation_display(self):
        """Update display during animation"""
        self.update_display()
        progress = (self.current_time / self.simulation_time[1]) * 100
        self.progress_var.set(progress)
    
    def force_recalculate(self):
        """Force recalculation of the system"""
        self.stop_animation()  # Stop animation when recalculating
        self.current_time = 0  # Reset to start
        self.manual_time_var.set(0)
        
        print("🔄 Recalculating system with new parameters...")
        self.solve_system()
        self.update_display()
        self.update_values_display()
        print("✅ System recalculated!")
    
    def reset_all(self):
        """Reset all parameters to default"""
        self.stop_animation()
        
        # Parameter defaults
        param_defaults = {
            'u': 0.05, 'a2': 0.01, 'a3': 0.01, 'a4': 0.008, 'a5': 0.008,
            'A1': 1.0, 'A2': 0.8, 'A3': 0.8
        }
        
        # Initial condition defaults
        init_defaults = {
            'init_h1': 0.1, 'init_h2': 0.05, 'init_h3': 0.05
        }
        
        # Reset all sliders
        for param, value in param_defaults.items():
            self.sliders[param]['var'].set(value)
            self.params[param] = value
            self.sliders[param]['label'].config(text=f"{value:.3f}")
        
        for param, value in init_defaults.items():
            self.sliders[param]['var'].set(value)
            tank_idx = int(param[-1]) - 1
            self.initial_conditions[tank_idx] = value
            self.sliders[param]['label'].config(text=f"{value:.3f}")
        
        # Reset animation controls
        self.current_time = 0
        self.manual_time_var.set(0)
        self.speed_var.set(1.0)
        self.animation_speed = 1.0
        self.speed_label.config(text="1.0x")
        self.progress_var.set(0)
        
        self.force_recalculate()
        print("✅ All parameters reset to defaults!")
    
    def update_values_display(self):
        """Update the current values display"""
        if self.solution is None:
            return
            
        time_idx = np.abs(self.solution.t - self.current_time).argmin()
        current_h1 = self.solution.y[0][time_idx]
        current_h2 = self.solution.y[1][time_idx]
        current_h3 = self.solution.y[2][time_idx]
        
        # Calculate flow rates
        flow_a2 = self.params['Smax'] * self.params['a2'] * np.sqrt(2 * self.params['g'] * current_h1)
        flow_a3 = self.params['Smax'] * self.params['a3'] * np.sqrt(2 * self.params['g'] * current_h1)
        flow_a4 = self.params['Smax'] * self.params['a4'] * np.sqrt(2 * self.params['g'] * current_h2)
        flow_a5 = self.params['Smax'] * self.params['a5'] * np.sqrt(2 * self.params['g'] * current_h3)
        
        text = (
            f"REAL-TIME VALUES:\n"
            f"Time: {self.current_time:6.1f} s\n"
            f"Status: {'▶ ANIMATING' if self.is_animating else '⏸ PAUSED'}\n"
            f"Speed: {self.animation_speed:.1f}x\n"
            f"Auto-recalc: {'✅ ON' if self.auto_recalculate else '❌ OFF'}\n\n"
            f"WATER LEVELS:\n"
            f"h1 (Tank 1): {current_h1:6.3f} m\n"
            f"h2 (Tank 2): {current_h2:6.3f} m\n"
            f"h3 (Tank 3): {current_h3:6.3f} m\n\n"
            f"FLOW RATES:\n"
            f"Input:    {self.params['u']:6.4f} m³/s\n"
            f"T1→T2:   {flow_a2:6.4f} m³/s\n"
            f"T1→T3:   {flow_a3:6.4f} m³/s\n"
            f"T2 Out:  {flow_a4:6.4f} m³/s\n"
            f"T3 Out:  {flow_a5:6.4f} m³/s\n\n"
            f"VALVE SETTINGS:\n"
            f"a2: {self.params['a2']:6.3f}\n"
            f"a3: {self.params['a3']:6.3f}\n"
            f"a4: {self.params['a4']:6.3f}\n"
            f"a5: {self.params['a5']:6.3f}\n"
        )
        
        self.values_text.delete('1.0', tk.END)
        self.values_text.insert('1.0', text)
    
    def update_display(self):
        """Update the matplotlib displays"""
        if self.solution is None:
            return
            
        # Find current state
        time_idx = np.abs(self.solution.t - self.current_time).argmin()
        current_h1 = self.solution.y[0][time_idx]
        current_h2 = self.solution.y[1][time_idx]
        current_h3 = self.solution.y[2][time_idx]
        
        # Update all plots
        self.draw_tanks(current_h1, current_h2, current_h3)
        self.draw_levels_plot()
        self.draw_h1_plot()
        
        # Update time display
        self.time_display.config(text=f"Time: {self.current_time:.1f} s")
        
        # Refresh canvas
        self.canvas.draw()
        self.update_values_display()
    
    def draw_tanks(self, h1, h2, h3):
        """Draw the three tanks with valves and pipes"""
        self.ax_tanks.clear()
        self.ax_tanks.set_xlim(-3, 3)
        self.ax_tanks.set_ylim(-0.5, 2.5)
        self.ax_tanks.set_aspect('equal')
        self.ax_tanks.set_title('Three-Tank System - Real Time View', fontweight='bold', fontsize=14)
        self.ax_tanks.set_xlabel('Position')
        self.ax_tanks.set_ylabel('Water Level (m)')
        self.ax_tanks.grid(True, alpha=0.3)
        
        # Tank positions: Tank 3 (left), Tank 1 (middle), Tank 2 (right)
        positions = [-1.8, 0, 1.8]
        heights = [h3, h1, h2]
        colors = ['coral', 'royalblue', 'limegreen']
        labels = ['Tank 3', 'Tank 1', 'Tank 2']
        
        tank_width = 0.6
        tank_height = 2.0
        
        for i, (x, h, color, label) in enumerate(zip(positions, heights, colors, labels)):
            # Tank outline
            self.ax_tanks.plot([x - tank_width/2, x - tank_width/2], [0, tank_height], 'k-', linewidth=3)
            self.ax_tanks.plot([x + tank_width/2, x + tank_width/2], [0, tank_height], 'k-', linewidth=3)
            self.ax_tanks.plot([x - tank_width/2, x + tank_width/2], [0, 0], 'k-', linewidth=3)
            self.ax_tanks.plot([x - tank_width/2, x + tank_width/2], [tank_height, tank_height], 'k-', linewidth=1, alpha=0.5)
            
            # Water
            if h > 0:
                water = plt.Rectangle((x - tank_width/2, 0), tank_width, h, 
                                    color=color, alpha=0.7, edgecolor='darkblue', linewidth=1)
                self.ax_tanks.add_patch(water)
                
                # Water level line
                self.ax_tanks.plot([x - tank_width/2, x + tank_width/2], [h, h], 
                                 'white', linewidth=2)
                
                # Level text
                self.ax_tanks.text(x, h + 0.1, f'{h:.3f} m', 
                                 ha='center', va='bottom', fontweight='bold', fontsize=10,
                                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            # Tank label
            self.ax_tanks.text(x, -0.3, label, ha='center', va='top', fontweight='bold', fontsize=11)
        
        # Draw pipes and valves
        self.draw_valves_and_pipes(h1, h2, h3)
    
    def draw_valves_and_pipes(self, h1, h2, h3):
        """Draw pipes and valves between tanks"""
        # Pipe from Tank 1 to Tank 2 (left)
        pipe_y = 1.5
        self.ax_tanks.plot([0, -1.8], [pipe_y, pipe_y], 'b-', linewidth=3, alpha=0.7)
        
        # Valve a2
        valve_x2 = -0.9
        valve_size = 0.15
        self.ax_tanks.plot([valve_x2, valve_x2], [pipe_y - valve_size, pipe_y + valve_size], 
                         'r-', linewidth=4, label=f'a2={self.params["a2"]:.3f}')
        self.ax_tanks.text(valve_x2, pipe_y + 0.2, f'a2\n{self.params["a2"]:.3f}', 
                         ha='center', va='bottom', fontweight='bold', fontsize=8,
                         bbox=dict(boxstyle='round', facecolor='red', alpha=0.7))
        
        # Pipe from Tank 1 to Tank 3 (right)
        self.ax_tanks.plot([0, 1.8], [pipe_y, pipe_y], 'b-', linewidth=3, alpha=0.7)
        
        # Valve a3
        valve_x3 = 0.9
        self.ax_tanks.plot([valve_x3, valve_x3], [pipe_y - valve_size, pipe_y + valve_size], 
                         'r-', linewidth=4, label=f'a3={self.params["a3"]:.3f}')
        self.ax_tanks.text(valve_x3, pipe_y + 0.2, f'a3\n{self.params["a3"]:.3f}', 
                         ha='center', va='bottom', fontweight='bold', fontsize=8,
                         bbox=dict(boxstyle='round', facecolor='red', alpha=0.7))
        
        # Outlet pipes and valves
        outlet_y = 0.2
        
        # Tank 2 outlet (a4)
        self.ax_tanks.plot([-1.8, -2.5], [outlet_y, outlet_y], 'g-', linewidth=2, alpha=0.7)
        self.ax_tanks.plot([-2.1, -2.1], [outlet_y - valve_size, outlet_y + valve_size], 
                         'orange', linewidth=4)
        self.ax_tanks.text(-2.3, outlet_y + 0.15, f'a4\n{self.params["a4"]:.3f}', 
                         ha='center', va='bottom', fontweight='bold', fontsize=8,
                         bbox=dict(boxstyle='round', facecolor='orange', alpha=0.7))
        
        # Tank 3 outlet (a5)
        self.ax_tanks.plot([1.8, 2.5], [outlet_y, outlet_y], 'g-', linewidth=2, alpha=0.7)
        self.ax_tanks.plot([2.1, 2.1], [outlet_y - valve_size, outlet_y + valve_size], 
                         'orange', linewidth=4)
        self.ax_tanks.text(2.3, outlet_y + 0.15, f'a5\n{self.params["a5"]:.3f}', 
                         ha='center', va='bottom', fontweight='bold', fontsize=8,
                         bbox=dict(boxstyle='round', facecolor='orange', alpha=0.7))
        
        # Input pipe to Tank 1
        input_y = 1.8
        self.ax_tanks.plot([-0.5, 0], [input_y, input_y], 'purple', linewidth=3, alpha=0.7)
        self.ax_tanks.text(-0.25, input_y + 0.1, f'u={self.params["u"]:.3f}', 
                         ha='center', va='bottom', fontweight='bold', fontsize=8,
                         bbox=dict(boxstyle='round', facecolor='purple', alpha=0.7))
        
        # Add legend for valves
        self.ax_tanks.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), 
                           ncol=3, fontsize=9)
    
    def draw_levels_plot(self):
        """Draw the water levels over time"""
        if self.solution is None:
            return
            
        self.ax_levels.clear()
        self.ax_levels.set_xlim(0, 100)
        self.ax_levels.set_ylim(0, 2.0)
        self.ax_levels.set_title('All Water Levels Over Time', fontweight='bold', fontsize=12)
        self.ax_levels.set_xlabel('Time (s)')
        self.ax_levels.set_ylabel('Water Level (m)')
        self.ax_levels.grid(True, alpha=0.3)
        
        # Find how much to plot based on current time
        time_idx = np.abs(self.solution.t - self.current_time).argmin()
        frames_to_plot = min(time_idx + 1, len(self.solution.t))
        
        # Plot all levels
        self.ax_levels.plot(self.solution.t[:frames_to_plot], self.solution.y[0][:frames_to_plot], 
                          'b-', label='h1 (Tank 1)', linewidth=2)
        self.ax_levels.plot(self.solution.t[:frames_to_plot], self.solution.y[1][:frames_to_plot], 
                          'g-', label='h2 (Tank 2)', linewidth=2)
        self.ax_levels.plot(self.solution.t[:frames_to_plot], self.solution.y[2][:frames_to_plot], 
                          'r-', label='h3 (Tank 3)', linewidth=2)
        
        # Current time marker
        current_h1 = self.solution.y[0][time_idx]
        current_h2 = self.solution.y[1][time_idx]
        current_h3 = self.solution.y[2][time_idx]
        
        self.ax_levels.axvline(x=self.current_time, color='k', linestyle='--', alpha=0.7)
        self.ax_levels.plot(self.current_time, current_h1, 'bo', markersize=6)
        self.ax_levels.plot(self.current_time, current_h2, 'go', markersize=6)
        self.ax_levels.plot(self.current_time, current_h3, 'ro', markersize=6)
        
        self.ax_levels.legend()
    
    def draw_h1_plot(self):
        """Draw individual h1 plot"""
        if self.solution is None:
            return
            
        self.ax_h1.clear()
        self.ax_h1.set_xlim(0, 100)
        self.ax_h1.set_ylim(0, 2.0)
        self.ax_h1.set_title('Tank 1 Water Level (h1)', fontweight='bold', fontsize=12)
        self.ax_h1.set_xlabel('Time (s)')
        self.ax_h1.set_ylabel('h1 Level (m)')
        self.ax_h1.grid(True, alpha=0.3)
        
        # Find how much to plot based on current time
        time_idx = np.abs(self.solution.t - self.current_time).argmin()
        frames_to_plot = min(time_idx + 1, len(self.solution.t))
        
        # Plot h1
        self.ax_h1.plot(self.solution.t[:frames_to_plot], self.solution.y[0][:frames_to_plot], 
                       'b-', linewidth=3, alpha=0.8, label='h1')
        
        # Current time marker
        current_h1 = self.solution.y[0][time_idx]
        self.ax_h1.axvline(x=self.current_time, color='k', linestyle='--', alpha=0.7)
        self.ax_h1.plot(self.current_time, current_h1, 'bo', markersize=8)
        
        # Current value text
        self.ax_h1.text(0.05, 0.95, f'Current h1: {current_h1:.3f} m', 
                       transform=self.ax_h1.transAxes, fontweight='bold',
                       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

# Create style for better looking buttons
def configure_styles():
    style = ttk.Style()
    style.configure('Accent.TButton', font=('Arial', 10, 'bold'), foreground='white', background='#0078D7')
    style.configure('TNotebook.Tab', font=('Arial', 9, 'bold'), padding=[10, 5])

# Run the application
if __name__ == "__main__":
    root = tk.Tk()
    configure_styles()
    app = FullyLinkedTankSystem(root)
    
    print("🚀 ENHANCED TANK SYSTEM STARTED!")
    print("=" * 70)
    print("🎮 NEW FEATURES:")
    print("   • VISUAL VALVES between tanks with real-time values")
    print("   • TABBED INTERFACE for better organization")
    print("   • INSTANT PARAMETER UPDATES without restarting")
    print("   • FLOW RATE CALCULATIONS displayed in real-time")
    print("   • IMPROVED LAYOUT and positioning")
    print("   • BETTER VALVE LABELS and pipe visualization")
    print("=" * 70)
    
    root.mainloop()